In [1]:
!pip install tabpfn catboost lightgbm xgboost scikit-learn pandas numpy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.8/218.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# --- 1. Load Data ---
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# IMPORTANT: Update with your actual target column names
TARGETS = ['Genetic Disorder', 'Disorder Subclass']
ID_COL = 'Patient Id'

# --- 2. Setup Features (NO DATA REMOVED) ---
X = train.drop(columns=TARGETS + [ID_COL], errors='ignore')
test_features = test.drop(columns=[ID_COL], errors='ignore')

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Encode Categoricals (Fill NaNs with 'Missing' so LabelEncoder works)
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    test_features[col] = test_features[col].fillna('Missing').astype(str)

    le = LabelEncoder()
    le.fit(list(X[col]) + list(test_features[col]))
    X[col] = le.transform(X[col])
    test_features[col] = le.transform(test_features[col])

# Note: We are intentionally NOT filling missing values in `num_cols`.
# XGBoost will natively learn the best way to handle these NaNs!

# --- 3. XGBoost Parameters (GPU Enabled) ---
# --- 3. XGBoost Parameters (GPU Enabled) ---
xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42
}

# --- 4. Cross Validation Loop ---
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
final_test_predictions = {}
target_encoders = {}

for target in TARGETS:
    print(f"\n{'='*40}")
    print(f"TRAINING XGBOOST ONLY FOR: {target}")
    print(f"{'='*40}")

    # We must filter out rows where the TARGET label itself is missing,
    # because a model cannot train if it doesn't know the answer.
    valid_indices = train[target].notna()
    X_valid = X[valid_indices].reset_index(drop=True)
    y_raw = train.loc[valid_indices, target].reset_index(drop=True)

    le_target = LabelEncoder()
    y_valid = pd.Series(le_target.fit_transform(y_raw))
    target_encoders[target] = le_target
    n_classes = len(le_target.classes_)

    oof_preds = np.zeros((len(X_valid), n_classes))
    test_preds = np.zeros((len(test_features), n_classes))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_valid, y_valid)):
        X_train, y_train = X_valid.iloc[train_idx], y_valid.iloc[train_idx]
        X_val, y_val = X_valid.iloc[val_idx], y_valid.iloc[val_idx]

        # Train XGBoost
        model_xgb = XGBClassifier(**xgb_params)
        model_xgb.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # Predict
        fold_val_preds = model_xgb.predict_proba(X_val)
        fold_test_preds = model_xgb.predict_proba(test_features)

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / N_SPLITS

        fold_acc = accuracy_score(y_val, np.argmax(fold_val_preds, axis=1))
        print(f"Fold {fold+1} Acc: {fold_acc:.4f}")

    final_oof_acc = accuracy_score(y_valid, np.argmax(oof_preds, axis=1))
    print(f"--> Final OOF Acc for {target}: {final_oof_acc:.5f}\n")
    final_test_predictions[target] = test_preds

# --- 5. Generate Submission ---
submission = pd.DataFrame({ID_COL: test[ID_COL]})
for target in TARGETS:
    submission[target] = target_encoders[target].inverse_transform(np.argmax(final_test_predictions[target], axis=1))

submission.to_csv('submission_xgb_only.csv', index=False)
print("Saved XGBoost predictions to submission_xgb_only.csv!")


TRAINING XGBOOST ONLY FOR: Genetic Disorder
Fold 1 Acc: 0.5677
Fold 2 Acc: 0.5775
Fold 3 Acc: 0.5756
Fold 4 Acc: 0.5744
Fold 5 Acc: 0.5613
--> Final OOF Acc for Genetic Disorder: 0.57130


TRAINING XGBOOST ONLY FOR: Disorder Subclass
Fold 1 Acc: 0.4077
Fold 2 Acc: 0.3917
Fold 3 Acc: 0.3821
Fold 4 Acc: 0.3821
Fold 5 Acc: 0.3769
--> Final OOF Acc for Disorder Subclass: 0.38810

Saved XGBoost predictions to submission_xgb_only.csv!


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

# --- 1. Load Data ---
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

TARGETS = ['Genetic Disorder', 'Disorder Subclass']
ID_COL = 'Patient Id'

# --- 2. Setup Features (NO DATA REMOVED) ---
X = train.drop(columns=TARGETS + [ID_COL], errors='ignore')
test_features = test.drop(columns=[ID_COL], errors='ignore')

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Encode Categoricals
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    test_features[col] = test_features[col].fillna('Missing').astype(str)

    le = LabelEncoder()
    le.fit(list(X[col]) + list(test_features[col]))
    X[col] = le.transform(X[col])
    test_features[col] = le.transform(test_features[col])

# --- 3. LightGBM Parameters (GPU Enabled) ---
lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'device': 'gpu',             # Use GPU
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1                # Silences warnings
}

# --- 4. Cross Validation Loop ---
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
final_test_predictions = {}
target_encoders = {}

for target in TARGETS:
    print(f"\n{'='*40}")
    print(f"TRAINING LIGHTGBM ONLY FOR: {target}")
    print(f"{'='*40}")

    valid_indices = train[target].notna()
    X_valid = X[valid_indices].reset_index(drop=True)
    y_raw = train.loc[valid_indices, target].reset_index(drop=True)

    le_target = LabelEncoder()
    y_valid = pd.Series(le_target.fit_transform(y_raw))
    target_encoders[target] = le_target
    n_classes = len(le_target.classes_)

    oof_preds = np.zeros((len(X_valid), n_classes))
    test_preds = np.zeros((len(test_features), n_classes))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_valid, y_valid)):
        X_train, y_train = X_valid.iloc[train_idx], y_valid.iloc[train_idx]
        X_val, y_val = X_valid.iloc[val_idx], y_valid.iloc[val_idx]

        # Train LightGBM
        model_lgb = LGBMClassifier(**lgb_params)
        model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])

        fold_val_preds = model_lgb.predict_proba(X_val)
        fold_test_preds = model_lgb.predict_proba(test_features)

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / N_SPLITS

        fold_acc = accuracy_score(y_val, np.argmax(fold_val_preds, axis=1))
        print(f"Fold {fold+1} Acc: {fold_acc:.4f}")

    final_oof_acc = accuracy_score(y_valid, np.argmax(oof_preds, axis=1))
    print(f"--> Final OOF Acc for {target}: {final_oof_acc:.5f}\n")
    final_test_predictions[target] = test_preds

# --- 5. Generate Submission ---
submission = pd.DataFrame({ID_COL: test[ID_COL]})
for target in TARGETS:
    submission[target] = target_encoders[target].inverse_transform(np.argmax(final_test_predictions[target], axis=1))

submission.to_csv('submission_lgb_only.csv', index=False)
print("Saved LightGBM predictions to submission_lgb_only.csv!")


TRAINING LIGHTGBM ONLY FOR: Genetic Disorder
Fold 1 Acc: 0.5670
Fold 2 Acc: 0.5757
Fold 3 Acc: 0.5573
Fold 4 Acc: 0.5648
Fold 5 Acc: 0.5666
--> Final OOF Acc for Genetic Disorder: 0.56628


TRAINING LIGHTGBM ONLY FOR: Disorder Subclass
Fold 1 Acc: 0.4025


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

# --- 1. Load Data ---
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

TARGETS = ['Genetic Disorder', 'Disorder Subclass']
ID_COL = 'Patient Id'

# --- 2. Setup Features (NO DATA REMOVED) ---
X = train.drop(columns=TARGETS + [ID_COL], errors='ignore')
test_features = test.drop(columns=[ID_COL], errors='ignore')

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Encode Categoricals
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    test_features[col] = test_features[col].fillna('Missing').astype(str)

    le = LabelEncoder()
    le.fit(list(X[col]) + list(test_features[col]))
    X[col] = le.transform(X[col])
    test_features[col] = le.transform(test_features[col])

# --- 3. CatBoost Parameters (GPU Enabled) ---
cat_params = {
    'iterations': 1500,
    'learning_rate': 0.05,
    'depth': 6,
    'task_type': 'GPU',          # Use GPU
    'random_seed': 42,
    'verbose': 0                 # Silences iteration output
}

# --- 4. Cross Validation Loop ---
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
final_test_predictions = {}
target_encoders = {}

for target in TARGETS:
    print(f"\n{'='*40}")
    print(f"TRAINING CATBOOST ONLY FOR: {target}")
    print(f"{'='*40}")

    valid_indices = train[target].notna()
    X_valid = X[valid_indices].reset_index(drop=True)
    y_raw = train.loc[valid_indices, target].reset_index(drop=True)

    le_target = LabelEncoder()
    y_valid = pd.Series(le_target.fit_transform(y_raw))
    target_encoders[target] = le_target
    n_classes = len(le_target.classes_)

    oof_preds = np.zeros((len(X_valid), n_classes))
    test_preds = np.zeros((len(test_features), n_classes))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_valid, y_valid)):
        X_train, y_train = X_valid.iloc[train_idx], y_valid.iloc[train_idx]
        X_val, y_val = X_valid.iloc[val_idx], y_valid.iloc[val_idx]

        # Train CatBoost
        model_cat = CatBoostClassifier(**cat_params)
        model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100, verbose=False)

        fold_val_preds = model_cat.predict_proba(X_val)
        fold_test_preds = model_cat.predict_proba(test_features)

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / N_SPLITS

        fold_acc = accuracy_score(y_val, np.argmax(fold_val_preds, axis=1))
        print(f"Fold {fold+1} Acc: {fold_acc:.4f}")

    final_oof_acc = accuracy_score(y_valid, np.argmax(oof_preds, axis=1))
    print(f"--> Final OOF Acc for {target}: {final_oof_acc:.5f}\n")
    final_test_predictions[target] = test_preds

# --- 5. Generate Submission ---
submission = pd.DataFrame({ID_COL: test[ID_COL]})
for target in TARGETS:
    submission[target] = target_encoders[target].inverse_transform(np.argmax(final_test_predictions[target], axis=1))

submission.to_csv('submission_cat_only.csv', index=False)
print("Saved CatBoost predictions to submission_cat_only.csv!")